## 向量的生成和检索

In [1]:
from langchain_community.embeddings import DashScopeEmbeddings
from dotenv import load_dotenv
from langchain_chroma import Chroma #引入向量数据库
load_dotenv()

# 向量生成
# 1. 原始内容
lines = []
with open('./data/couplettest.csv', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            lines.append(line)        
print(lines[:5])
    
# 2. 向量模型|嵌入模型
embedding_model = DashScopeEmbeddings(model="text-embedding-v1") #使用百炼平台的向量模型

# 3. 生成向量
# 4. 向量存储
vector_store = Chroma(
    persist_directory="./data/vector_db",#向量存储的位置, 默认是以sqlite文件方式存储
    embedding_function=embedding_model, #指定嵌入模型
    collection_name="couplet" #指定集合名称
)

print(f"开始添加 {len(lines)} 条数据到向量数据库中...")
vector_store.add_texts(lines)
print("添加完成")

C:\Users\21566\AppData\Local\Temp\ipykernel_27136\1607091594.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings
c:\Users\21566\anaconda3\envs\ai-langchain1.0\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


['\ufeff晚风摇树树还挺 晨露润花花更红', '愿景天成无墨迹 万方乐奏有于阗', '丹枫江冷人初去 绿柳堤新燕复来', '闲来野钓人稀处 兴起高歌酒醉中', '投石向天跟命斗 闭门问卷与时争']
开始添加 1673 条数据到向量数据库中...
添加完成


In [2]:
# 5. 向量检索
results = vector_store.similarity_search("春眠不觉晓", k=3)
for r in results:
    print(r)

page_content='不负春风催永昼 堪憎秋雨洒通宵'
page_content='闻香听雨知春暖 惊梦伤风晓夜珊'
page_content='不睡其心枉睡目 先宽彼带再宽衣'


In [6]:
results = vector_store.similarity_search_with_score("春眠不觉晓", k=3)

context = []
for doc, score in results:
    context.append(doc.page_content)

In [7]:
context

['不负春风催永昼 堪憎秋雨洒通宵', '闻香听雨知春暖 惊梦伤风晓夜珊', '不睡其心枉睡目 先宽彼带再宽衣']

In [4]:
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4o")

In [5]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
     你是一个专业的对联大师，你的任务是根据用户给出的上联，生成一句下联。
     回答时，可以参考下面的内容：
     {context}
     请用中文回答。
     """),
    ("user", "{text}")
])

In [8]:
prompt = prompt_template.invoke({"context": context, "text": "春眠不觉晓"})
prompt
                                 

c:\Users\21566\anaconda3\envs\ai-langchain1.0\Lib\site-packages\langsmith\_internal\_agent_addressing.py:61: LangSmithBetaWarning: Agent addressing (`agent_id` / `agent_environment`) is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


ChatPromptValue(messages=[SystemMessage(content="\n     你是一个专业的对联大师，你的任务是根据用户给出的上联，生成一句下联。\n     回答时，可以参考下面的内容：\n     ['不负春风催永昼 堪憎秋雨洒通宵', '闻香听雨知春暖 惊梦伤风晓夜珊', '不睡其心枉睡目 先宽彼带再宽衣']\n     请用中文回答。\n     ", additional_kwargs={}, response_metadata={}), HumanMessage(content='春眠不觉晓', additional_kwargs={}, response_metadata={})])

In [9]:
result = model.invoke(prompt)
print(result.content)

处处闻啼鸟
